# Lezione 8: Treemap e tabelle pivot (pivot chart)

**Avviso onesto prima di iniziare:** questa lezione è più breve delle precedenti. Il materiale sorgente di questo argomento è una guida concettuale (un PDF), senza un notebook d'esercizio proprio né un caso di business reale del corso — a differenza, ad esempio, del dataset delle compagnie aeree usato nella lezione precedente. Ciononostante, vale la pena conoscere questi due strumenti: ampliano la tua cassetta degli attrezzi per due situazioni molto specifiche che i grafici che già conosci non coprono bene.

Un **treemap** ti serve quando hai dati gerarchici (categorie dentro categorie) e vuoi vedere a colpo d'occhio quale parte pesa di più. Una **tabella pivot (pivot chart)** ti serve quando hai una tabella lunga e disordinata e devi riassumerla incrociando due variabili contemporaneamente.

## 1. Treemap: l'analogia dell'armadio

Pensa a un armadio con cassetti, e dentro ogni cassetto, degli scomparti.

- L'**armadio completo** è il tuo dataset.
- Ogni **cassetto** è una categoria (ad esempio, "Elettronica").
- Ogni **scomparto** dentro il cassetto è una sottocategoria (ad esempio, "Laptop").

La dimensione di ogni scomparto ti dice, a colpo d'occhio, quanto spazio occupa quella categoria nel totale — senza bisogno di leggere un solo numero.

Un treemap rappresenta dati gerarchici con rettangoli annidati: ogni ramo della gerarchia è un rettangolo, diviso a sua volta in rettangoli più piccoli per i suoi sotto-rami. È molto usato nell'analisi di business (vendite per categoria e sottocategoria), nella finanza (composizione di un portafoglio), o nella gestione dei file (spazio occupato per cartella).

Lo costruisci con `px.treemap()`. Ti serve un dataframe con almeno tre colonne: la categoria, la sottocategoria, e un valore numerico che determina la dimensione di ogni rettangolo.

In [ ]:
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Dati di esempio: vendite per categoria e sottocategoria
data = {
    'Category': ['Electronics', 'Electronics', 'Electronics',
                 'Furniture', 'Furniture', 'Furniture',
                 'Clothing', 'Clothing', 'Clothing'],
    'Subcategory': ['Laptops', 'Smartphones', 'Tablets',
                     'Chairs', 'Tables', 'Sofas',
                     'Men', 'Women', 'Kids'],
    'Sales': [120000, 80000, 30000,
              50000, 40000, 20000,
              70000, 90000, 40000]
}
df = pd.DataFrame(data)
df

In [ ]:
# Creiamo il treemap
fig = px.treemap(
    df,
    path=['Category', 'Subcategory'],  # Definisce la gerarchia: prima categoria, poi sottocategoria
    values='Sales',                    # La dimensione di ogni rettangolo
    title='Sales Data Treemap'
)
fig.show()

Il parametro `path` è la chiave: definisce l'ordine della gerarchia, dal più generale al più specifico. Qui, ogni categoria ("Electronics", "Furniture", "Clothing") diventa un rettangolo grande, e al suo interno appaiono rettangoli più piccoli per ogni sottocategoria, con la dimensione proporzionale a `Sales`. A colpo d'occhio vedi che "Women" è la sottocategoria con più vendite dentro "Clothing", senza leggere una sola cifra.

## 2. Tabelle pivot con pandas: l'analogia di Excel

Se hai mai usato le tabelle pivot in Excel (trascinare un campo su "righe", un altro su "colonne" e un altro su "valori"), un pivot chart in Python è esattamente questo, ma programmatico: invece di trascinare con il mouse, scrivi una riga di codice che dice a pandas come incrociare le tue colonne.

Prima di creare il grafico, devi riassumere i dati. Per questo usi `pivot_table()` di pandas, che riorganizza una tabella lunga incrociando una o più colonne.

Simuliamo vendite di prodotti IT per trimestre, con colonne `Date`, `Category`, `Subcategory` e `Sales` (come nell'esempio del PDF).

In [ ]:
# Dati di esempio: vendite di prodotti IT per trimestre
np.random.seed(42)

quarters = ['Q1', 'Q2', 'Q3', 'Q4']
categories = ['Desktops', 'Laptops', 'Peripherals', 'Software']
subcategories = ['Accessories', 'Components', 'Software Suites']

n = 200
df_it = pd.DataFrame({
    'Date': np.random.choice(quarters, n),
    'Category': np.random.choice(categories, n),
    'Subcategory': np.random.choice(subcategories, n),
    'Sales': np.random.randint(1000, 5000, n)
})
df_it.head()

In [ ]:
# Creiamo la tabella pivot:
# - index: cosa va nelle righe (qui, il trimestre)
# - columns: cosa va nelle colonne (qui, categoria e sottocategoria)
# - values: quale colonna viene aggregata
# - aggfunc: come si aggrega (qui, somma)
pivot_table = df_it.pivot_table(
    index='Date',
    columns=['Category', 'Subcategory'],
    values='Sales',
    aggfunc='sum'
)
pivot_table

Il risultato è una tabella dove ogni riga è un trimestre e ogni colonna è una combinazione di categoria e sottocategoria, con la somma delle vendite corrispondente. È esattamente ciò che faresti in Excel trascinando campi nelle zone di righe, colonne e valori.

## 3. Dalla tabella pivot al grafico (pivot chart)

Una volta ottenuta la tabella pivot, crearne il grafico è diretto: pandas ti permette di chiamare `.plot()` direttamente su di essa.

In [ ]:
# Creiamo il grafico della tabella pivot come barre
pivot_table.plot(kind='bar', figsize=(14, 8))
plt.title('Sales Summary of IT Products by Category and Subcategory')
plt.xlabel('Quarters')
plt.ylabel('Total Sales')
plt.grid(False)
plt.legend(title=('Category', 'Subcategory'), bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

Ogni gruppo di barre rappresenta un trimestre, e dentro ogni gruppo c'è una barra per ogni combinazione di categoria e sottocategoria. `bbox_to_anchor` e `loc` in `plt.legend()` spostano la legenda fuori dall'area del grafico, così non copre le barre quando ci sono molte combinazioni.

## Punti chiave

- Un treemap mostra gerarchie (categoria → sottocategoria) con rettangoli annidati, dove la dimensione comunica la grandezza senza bisogno di leggere numeri.
- In `px.treemap()`, il parametro `path` definisce l'ordine della gerarchia (dal generale allo specifico) e `values` definisce la dimensione di ogni rettangolo.
- Una tabella pivot (`pivot_table()`) riassume una tabella lunga incrociando righe (`index`), colonne (`columns`) e un valore aggregato (`values` + `aggfunc`) — è la versione programmatica delle tabelle pivot di Excel.
- Una volta ottenuta la tabella pivot, `.plot(kind='bar')` la trasforma direttamente in un grafico a barre raggruppate.
- Questa coppia di strumenti non sostituisce ciò che già sai (scatter, bar, pie...): li usi specificamente quando i tuoi dati sono gerarchici o quando devi incrociare due variabili categoriche contemporaneamente.